<a href="https://colab.research.google.com/github/somendrew/RAG_System/blob/main/Basic_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to Retrieval Augmented Generation (RAG)

What RAG solves: Large Language Models (LLMs) often lack access to private, recent, or domain-specific data, leading to hallucinations or inaccurate responses. RAG addresses this by retrieving relevant text snippets from an external knowledge base (your own corpus) at query time. These retrieved snippets are then provided as context to the LLM, enabling the model to generate answers grounded in real, up-to-date information rather than relying solely on its pre-trained memory.

The RAG pipeline typically consists of two main phases:

## Indexing Phase (Offline - Performed once or periodically)
1.  **Load Documents**: Gather your source data (e.g., Wikipedia articles, internal documents, web pages).
2.  **Split into Chunks**: Break down the larger documents into smaller, manageable chunks of text.
3.  **Embed Chunks**: Convert these text chunks into numerical vector representations (embeddings).
4.  **Store Vectors in a Vector Database**: Save these embeddings in a specialized database for efficient similarity search.

## Retrieval + Generation Phase (Online - Performed per user query)
1.  **Embed the User's Question**: Convert the user's query into a vector embedding.
2.  **Similarity Search in Vector Database**: Use the query's embedding to find the most semantically similar chunks in the vector database.
3.  **Get Top-K Chunks**: Retrieve the top 'k' most relevant chunks.
4.  **Stuff into Prompt**: Insert these retrieved chunks into the LLM's prompt as context.
5.  **LLM Generates Answer**: The LLM uses the provided context to formulate an informed and accurate response.

In [52]:
from google.colab import userdata

# Retrieve the OpenAI API key from Colab's secrets manager.
# The key should be stored under the name 'OPENAI_API_KEY'.
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

In [42]:
# Install necessary Python libraries for LangChain, OpenAI integrations, ChromaDB, Wikipedia, and Tiktoken.
# - `langchain`: The core framework for building LLM applications.
# - `langchain-openai`: Integrations for OpenAI models (LLMs and Embeddings).
# - `langchain-community`: Community integrations for various data loaders, utilities, etc.
# - `langchain-chroma`: Integration for Chroma vector database.
# - `chromadb`: The Chroma vector database client.
# - `wikipedia`: Python wrapper for the Wikipedia API (used by WikipediaLoader).
# - `tiktoken`: OpenAI's tokenizer for counting tokens.
# - `langchain-text-splitters`: Utilities for splitting text into chunks.
!pip install -q langchain langchain-openai langchain-community langchain-chroma chromadb wikipedia tiktoken langchain-text-splitters

# 1. Data Loading: Fetching Information from Wikipedia

In this section, we'll load our corpus from Wikipedia. We define a utility function to fetch content for specified topics from Wikipedia, ensuring we get a structured document that can be processed by our RAG pipeline. This simulates obtaining external knowledge that our LLM will later use.

In [53]:
import requests
from langchain_core.documents import Document

def fetch_wikipedia_page(title: str) -> Document:
    """
    Fetches a Wikipedia page's extract as a LangChain Document.

    Args:
        title (str): The title of the Wikipedia page to fetch.

    Returns:
        Document: A LangChain Document object containing the page content and metadata.
    """
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",       # Action to perform: query the API.
        "prop": "extracts",      # Request page extracts.
        "explaintext": True,     # Return plain text instead of HTML.
        "titles": title,         # The title of the page to query.
        "format": "json",        # Response format: JSON.
        "redirects": 1,          # Resolve redirects.
    }
    headers = {"User-Agent": "RAG-Learning-Notebook/1.0 (educational use)"}

    resp = requests.get(url, params=params, headers=headers, timeout=10)
    resp.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx).
    data = resp.json()

    pages = data["query"]["pages"]
    page = next(iter(pages.values()))
    text = page.get("extract", "")

    return Document(page_content=text, metadata={"title": page.get("title", title), "source": url})

topics = ["Solar System","Moon"]

docs = [fetch_wikipedia_page(t) for t in topics]

for d in docs:
    print(d.metadata, "-", len(d.page_content), "chars")

{'title': 'Solar System', 'source': 'https://en.wikipedia.org/w/api.php'} - 61733 chars
{'title': 'Moon', 'source': 'https://en.wikipedia.org/w/api.php'} - 89900 chars


In [44]:
# Display the first 500 characters of the content of the first document (`docs[0]`)
# to get a quick preview of the loaded data.
docs[0].page_content[:500]

'The Solar System is the gravitationally bound system of the Sun and the masses that orbit it, most prominently its eight planets, of which Earth is one. The Solar System is an isolated single-star planetary system (not part of a larger star system) within the Milky Way Galaxy. The system formed about 4.6 billion years ago when a dense region of a molecular cloud collapsed, creating the Sun and a protoplanetary disc from which the orbiting bodies assembled.\nThe Sun accounts for 99.86% of the Sola'

# 2. Text Chunking: Preparing Documents for Embeddings

Large documents can be too extensive for an embedding model or an LLM's context window. This section focuses on splitting our loaded Wikipedia documents into smaller, overlapping chunks. This process is crucial for effective retrieval, as it ensures that relevant information can be pinpointed and provided to the LLM without exceeding token limits.

In [54]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the RecursiveCharacterTextSplitter.
# This splitter attempts to split by paragraphs, then sentences, then words, falling back to characters.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, # Each chunk will aim to be 1000 characters long.
    chunk_overlap=150, # Chunks will overlap by 150 characters to maintain context between them.
)

chunks = splitter.split_documents(docs)

print(f"Split into {len(chunks)} chunks")

Split into 251 chunks


In [46]:
# Display the first chunk to inspect its content and metadata.
# This helps verify that the chunking process has worked as expected.
chunks[0]

Document(metadata={'title': 'Solar System', 'source': 'https://en.wikipedia.org/w/api.php'}, page_content="The Solar System is the gravitationally bound system of the Sun and the masses that orbit it, most prominently its eight planets, of which Earth is one. The Solar System is an isolated single-star planetary system (not part of a larger star system) within the Milky Way Galaxy. The system formed about 4.6 billion years ago when a dense region of a molecular cloud collapsed, creating the Sun and a protoplanetary disc from which the orbiting bodies assembled.\nThe Sun accounts for 99.86% of the Solar System's total mass. Inside the Sun's core, hydrogen is fused into helium, releasing energy that is emitted through the Sun's photosphere. This creates the heliosphere and a decreasing temperature gradient across the Solar System.")

# 3. Embeddings and Vector Store: Creating a Searchable Knowledge Base

Once documents are chunked, they need to be converted into numerical representations (embeddings) that capture their semantic meaning. These embeddings are then stored in a vector database, which allows for fast and efficient similarity searches. When a user asks a question, its embedding can be used to find the most relevant document chunks.

In [55]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_chroma import Chroma

# Initialize the OpenAI Embeddings model.
# We use 'text-embedding-3-small' for cost-effectiveness and good performance.
# The OPENAI_API_KEY is retrieved from Colab secrets.
embeddings = OpenAIEmbeddings(model = "text-embedding-3-small", api_key = OPENAI_API_KEY)

# Create a Chroma vector store from the document chunks.
# - `documents`: The list of text chunks to embed and store.
# - `embedding`: The embedding model to use for converting text to vectors.
# - `persist_directory`: Specifies a local directory to save the Chroma database,
#   so it can be reloaded later without re-embedding.
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

# Create a retriever from the vector store.
# The retriever is responsible for fetching relevant documents (chunks) given a query.
# `search_kwargs={"k": 4}` means it will retrieve the top 4 most similar chunks.
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# 4. Building the RAG Chain with LangChain Expression Language (LCEL)

This section constructs the complete RAG (Retrieval Augmented Generation) chain using LangChain Expression Language (LCEL). This chain orchestrates the flow from receiving a user question, retrieving relevant context, forming a prompt, and finally generating an answer using an LLM. LCEL provides a flexible and composable way to build complex chains.

In [56]:
from langchain_openai.embeddings.base import MAX_TOKENS_PER_REQUEST
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Initialize the ChatOpenAI language model.
# - `model`: Specifies the LLM to use (e.g., 'gpt-4o-mini').
# - `temperature`: Controls the randomness of the output (0 means deterministic).
# - `api_key`: Your OpenAI API key.
# - `max_tokens`: Maximum number of tokens in the generated response.
llm = ChatOpenAI(
    model = "gpt-4o-mini",
    temperature = 0,
    api_key = OPENAI_API_KEY,
    max_tokens = 300,
    use_responses_api= False,
    )

# Define the chat prompt template for the RAG chain.
# It instructs the LLM to answer using ONLY the context below.
# If the answer isn't in the context, say you don't know.
prompt = ChatPromptTemplate.from_template(
    """
    Answer the question using ONLY the context below.
    If the answer isn't in the context, say you don't know.


    Context : {context}
    Question : {question}
    Answer :
    """
)

# Construct the RAG chain using LCEL.
# 1. `{"context": retriever, "question": RunnablePassthrough()}`:
#    - "context": The retriever takes the incoming question, fetches relevant chunks,
#      and passes them as 'context' to the next step.
#    - "question": `RunnablePassthrough()` takes the original question and passes it
#      through as 'question' to the next step (the prompt).
# 2. `| prompt`: The output from the previous step (context and question) is fed into the prompt template.
# 3. `| llm`: The formatted prompt is sent to the LLM for generation.
# 4. `| StrOutputParser()`: The raw LLM output is parsed into a simple string.
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [57]:
# Test the RAG chain with a question.
# This question is designed to check if the model correctly identifies when information is not in the context.
question = "How many planets are the in the earth?"

# Invoke the RAG chain with the question.
answer = rag_chain.invoke(question)

# Print the generated answer.
print(answer)

I don't know.


In [50]:
# Test the RAG chain with another question.
# This question is hypothetical and likely not directly answerable from the factual Wikipedia context,
# testing the model's ability to say 'I don't know' when context is insufficient.
question = "What will happen to solar system if we increase gravity by 0.1?"

# Invoke the RAG chain with the question.
answer = rag_chain.invoke(question)

# Print the generated answer.
print(answer)

I don't know.


In [51]:
# Test the RAG chain with a question that should be directly answerable from the loaded Wikipedia context.
question = "What is the mass of sun as compared to solar system?"

# Invoke the RAG chain with the question.
answer = rag_chain.invoke(question)

# Print the generated answer.
print(answer)

The Sun accounts for 99.86% of the Solar System's total mass.
